<a href="https://colab.research.google.com/github/Ferreira3/pipeline-de-etl-com-python/blob/main/pipeline_de_etl_com_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Bootcamp Santander 2025**
# **Explorando IA Generativa em um Pipeline de ETL com Python**

# **Enriquecendo descrições de produtos de um E-Commerce com IA Generativa**

Este projeto demonstra a aplicação de um pipeline de ETL (Extract, Transform, Load) utilizando Python para aprimorar descrições de produtos em uma plataforma de e-commerce fictícia (Serverest API). O objetivo principal é enriquecer a qualidade das descrições dos produtos, tornando-as mais detalhadas e atrativas, através da integração de Inteligência Artificial Generativa.

# **Pré-requisitos**
- Criar produtos teste para realizar a etapa de Extract
- Criar usuário
- Fazer login
- Guardar o token de autenticação

(A API utilizada no projeto não guarda os produtos cadastrados e token de autenticação por muito tempo.)

In [ ]:
import requests

api_url = 'https://serverest.dev'
test_products = [
    ["Mouse G305 Wirelesss Logitech", 260.00, "Informática", "10"],
    ["Smart TV 50 polegadas Samsung", 2280.00, "TV e Vídeo", "10"],
    ["Headset Cloud III Hyperx", 386.00, "Fone de ouvido", "10"],
    ["Fritadeira Air Fryer 4L Mondial", 236.00, "Eletroportáteis", "10"],
    ["Placa de Vídeo RTX 3060 NVIDIA", 1850.00, "Informática", "10"]
]
product_ids = []

# Criando usuário.
response_signup = requests.post(f"{api_url}/usuarios", json={
        "nome": "Zezão da Massa",
        "email": "zezao@bootcampsantander.com",
        "password": "teste",
        "administrador": "true"
  })
print("Usuário criado com sucesso!") if response_signup.status_code == 201 else print(f"Usuário não criado! Erro: {response_signup.status_code}")

# Fazendo login.
response_login = requests.post(f"{api_url}/login", json={
        "email": "zezao@bootcampsantander.com",
        "password": "teste"
    })
print("Login realizado com sucesso!") if response_login.status_code == 200 else print(f"Login não realizado! Erro: {response_login.status_code}")

# Guardando token de autorização gerado e confirmando a autenticação.
user_data = response_login.json()
auth_token = {"authorization":user_data["authorization"]}
response_token = requests.get(f"{api_url}", headers=auth_token)
print("Token autenticado com sucesso!") if response_token.status_code == 200 else print(f"Token não autenticado! Erro: {response_token.status_code}")

# Cadastrando 5 produtos teste e guardando os IDs numa lista.
for product in test_products:
  response_product = requests.post(f"{api_url}/produtos", headers=auth_token, json={
      "nome": product[0],
      "preco": product[1],
      "descricao": product[2],
      "quantidade": product[3]
  })
  product_data = response_product.json()
  if response_product.status_code == 201:
    product_ids.append(product_data["_id"])
    print(f"{product[0]} foi cadastrado com sucesso!")
  else:
    print(f"Produto não cadastrado! Erro: {response_product.status_code}")

# **E**xtract
Trazendo as informações dos produtos da API para uma lista via ID.

In [ ]:
products = []

for id in product_ids:
  response_product = requests.get(f"{api_url}/produtos/{id}")
  products.append(response_product.json())

for product in products:
  print(product)

# **T**ransform

Utilizando a API do OpenAI para gerar uma descrição mais detalhada e chamativa para cada produto.

In [ ]:
!pip install openai

In [ ]:
import openai

OPENAI_API_KEY = "SUA CHAVE AQUI"
client = openai.OpenAI(api_key=OPENAI_API_KEY)

# Gerando a descrição mais detalhada.
def generate_ai_description(product):
  completion = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
      {
          "role": "system",
          "content": "Você é um especialista em vendas online."
      },
      {
          "role": "user",
          "content": f"Crie uma descrição chamativa e interessante de no máximo 150 caracteres para o produto {product["nome"]} da categoria {product["descricao"]}. (Não inclua o nome do produto na descrição)"
      }
    ]
  )
  return completion.choices[0].message.content.strip('"')

# Atualizando a lista de produtos com as novas descrições.
for product in products:
  new_description = generate_ai_description(product)
  print(new_description)
  product["descricao"] = new_description

# **L**oad

Atualizando os produtos na API com as novas descrições via ID.

In [ ]:
# Enviando as descrições atualizadas para a API.
for product in products:
  response_product = requests.put(f"{api_url}/produtos/{product["_id"]}", headers=auth_token, json={
  "nome": product["nome"],
  "preco": product["preco"],
  "descricao": product["descricao"],
  "quantidade": product["quantidade"]
  })
  if response_product.status_code == 200:
    print(f"Produto \'{product["nome"]}\' atualizado com sucessso!")
  else:
    print(f"Produto \'{product["nome"]}\' não atualizado, erro:{response_product.status_code}")